In [ ]:
import socket
import jetbot

print("Running on:", socket.gethostname())
print("JetBot library:", jetbot.__file__)

In [ ]:
import time
import cv2
import numpy as np
import traitlets
import ipywidgets as widgets

from IPython.display import display
from jetbot import Camera, Robot, bgr8_to_jpeg

robot = Robot()
robot.stop()

camera = Camera.instance(width=224, height=224)

live_view = widgets.Image(format="jpeg", width=336, height=336)

camera_link = traitlets.dlink(
    (camera, "value"),
    (live_view, "value"),
    transform=bgr8_to_jpeg
)

display(live_view)

In [ ]:
latest = {"sample": None}

def receive_frame(change):
    latest["sample"] = (change["new"], time.monotonic())

camera.observe(receive_frame, names="value")

In [ ]:
BLACK_THRESHOLD = 70  # Initial guess; tune while stationary

def detect_line(frame):
    overlay = frame.copy()
    height, width = frame.shape[:2]

    # Inspect the floor ahead, excluding the very bottom edge.
    y0 = int(height * 0.60)
    y1 = int(height * 0.85)
    region = frame[y0:y1, :]

    gray = cv2.cvtColor(region, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    # Dark pixels become white in the detection mask.
    _, mask = cv2.threshold(
        gray, BLACK_THRESHOLD, 255, cv2.THRESH_BINARY_INV
    )

    # Compatible with OpenCV versions returning 2 or 3 values.
    contours = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )[-2]

    centre_x = width // 2
    cv2.rectangle(overlay, (0, y0), (width - 1, y1), (255, 0, 0), 1)
    cv2.line(overlay, (centre_x, y0), (centre_x, y1), (0, 255, 255), 1)

    error = None

    if contours:
        contour = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(contour)
        area_fraction = area / float(mask.size)
        moments = cv2.moments(contour)

        # Reject tiny specks and very large dark regions.
        if 0.005 < area_fraction < 0.35 and moments["m00"] > 0:
            cx = int(moments["m10"] / moments["m00"])
            cy = int(moments["m01"] / moments["m00"]) + y0

            error = (cx - centre_x) / float(width / 2)
            cv2.circle(overlay, (cx, cy), 5, (0, 255, 0), -1)

    label = "LINE LOST" if error is None else "error: %+.2f" % error
    cv2.putText(
        overlay, label, (5, 20),
        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 255), 1
    )

    return error, overlay, mask

In [ ]:
overlay_view = widgets.Image(format="jpeg", width=336, height=336)
mask_view = widgets.Image(format="jpeg", width=336, height=90)

display(widgets.HBox([overlay_view, mask_view]))

def preview(seconds=20):
    robot.stop()
    end = time.monotonic() + seconds

    while time.monotonic() < end:
        sample = latest["sample"]

        if sample is None:
            time.sleep(0.05)
            continue

        frame, captured_at = sample

        if time.monotonic() - captured_at > 0.5:
            print("Camera frames stopped arriving.")
            break

        error, overlay, mask = detect_line(frame)
        overlay_view.value = bgr8_to_jpeg(overlay)
        mask_view.value = bgr8_to_jpeg(
            cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
        )

        time.sleep(0.1)

preview()